# # FAISSインデックス確認ノートブック
# 
# このノートブックでは、作成されたFAISSインデックスの詳細情報を確認し、検索機能をテストします。

## 1. 必要なライブラリのインポート

In [2]:
!pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


In [3]:
import faiss
import pickle
import numpy as np
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureOpenAIEmbeddings
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pformat

In [6]:
# 環境変数読み込み
load_dotenv()

True

## 2. インデックスファイルの基本情報確認

In [7]:
def inspect_faiss_basic_info(storage_path="storage"):
    """FAISSインデックスの基本情報を表示"""
    
    print("=== FAISSインデックス基本情報 ===\n")
    
    # FAISSファイル確認
    faiss_file = f"{storage_path}/index.faiss"
    pkl_file = f"{storage_path}/index.pkl"
    
    print(f"📁 ファイル存在確認:")
    print(f"  - {faiss_file}: {'✅ 存在' if os.path.exists(faiss_file) else '❌ 不存在'}")
    print(f"  - {pkl_file}: {'✅ 存在' if os.path.exists(pkl_file) else '❌ 不存在'}")
    
    if not os.path.exists(faiss_file) or not os.path.exists(pkl_file):
        return None, None
    
    # FAISSインデックス読み込み
    index = faiss.read_index(faiss_file)
    
    print(f"\n📊 FAISSインデックス詳細:")
    print(f"  - インデックスタイプ: {type(index).__name__}")
    print(f"  - ベクトル次元数: {index.d}")
    print(f"  - 登録済みベクトル数: {index.ntotal}")
    print(f"  - メトリック: {index.metric_type}")
    
    if hasattr(index, 'is_trained'):
        print(f"  - 学習済み: {index.is_trained}")
    
    # ピクルファイル読み込み
    with open(pkl_file, 'rb') as f:
        pkl_data = pickle.load(f)
    
    print(f"\n📦 ピクルファイル内容:")
    if isinstance(pkl_data, dict):
        for key, value in pkl_data.items():
            print(f"  - {key}: {type(value).__name__}")
            if hasattr(value, '__len__') and key != 'embeddings':
                try:
                    print(f"    長さ: {len(value)}")
                except:
                    pass
    
    return index, pkl_data

In [8]:
# 基本情報確認実行
faiss_index, pkl_data = inspect_faiss_basic_info()

=== FAISSインデックス基本情報 ===

📁 ファイル存在確認:
  - storage/index.faiss: ✅ 存在
  - storage/index.pkl: ✅ 存在

📊 FAISSインデックス詳細:
  - インデックスタイプ: IndexFlatL2
  - ベクトル次元数: 1024
  - 登録済みベクトル数: 10
  - メトリック: 1
  - 学習済み: True

📦 ピクルファイル内容:


## 3. ドキュメント内容の詳細確認

In [9]:
def load_and_inspect_documents(storage_path="storage"):
    """ドキュメント内容を詳細確認"""
    
    try:
        # ダミー埋め込みでドキュメントのみ確認
        class DummyEmbeddings:
            def embed_documents(self, texts):
                return [[0.0] * 1536 for _ in texts]
            def embed_query(self, text):
                return [0.0] * 1536
        
        vectorstore = FAISS.load_local(storage_path, DummyEmbeddings(), allow_dangerous_deserialization=True)
        
        print("🔍 ドキュメント詳細情報:")
        
        if hasattr(vectorstore, 'docstore') and hasattr(vectorstore.docstore, '_dict'):
            docs = vectorstore.docstore._dict
            print(f"  - 総ドキュメント数: {len(docs)}")
            
            # ドキュメント統計
            doc_lengths = [len(doc.page_content) for doc in docs.values()]
            print(f"  - 平均文字数: {np.mean(doc_lengths):.0f}")
            print(f"  - 最小文字数: {np.min(doc_lengths)}")
            print(f"  - 最大文字数: {np.max(doc_lengths)}")
            
            # ドキュメント長分布をDataFrameで表示
            df_stats = pd.DataFrame({
                'ドキュメントID': list(docs.keys()),
                '文字数': [len(doc.page_content) for doc in docs.values()],
                'メタデータ数': [len(doc.metadata) if doc.metadata else 0 for doc in docs.values()]
            })
            
            print(f"\n📊 ドキュメント統計:")
            print(df_stats.describe())
            
            # 最初の3件をサンプル表示
            print(f"\n📄 ドキュメントサンプル:")
            for i, (doc_id, doc) in enumerate(list(docs.items())[:3]):
                print(f"\n  📑 ドキュメント {i+1}:")
                print(f"     ID: {doc_id}")
                print(f"     文字数: {len(doc.page_content)}")
                print(f"     内容: {doc.page_content[:200]}...")
                if doc.metadata:
                    print(f"     メタデータ: {doc.metadata}")
            
            return docs, df_stats
            
    except Exception as e:
        print(f"❌ ドキュメント確認エラー: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# ドキュメント確認実行
documents, doc_stats = load_and_inspect_documents()

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


🔍 ドキュメント詳細情報:
  - 総ドキュメント数: 10
  - 平均文字数: 917
  - 最小文字数: 547
  - 最大文字数: 998

📊 ドキュメント統計:
              文字数  メタデータ数
count   10.000000    10.0
mean   917.200000     1.0
std    137.493273     0.0
min    547.000000     1.0
25%    929.750000     1.0
50%    970.500000     1.0
75%    986.250000     1.0
max    998.000000     1.0

📄 ドキュメントサンプル:

  📑 ドキュメント 1:
     ID: aa99fa6a-7f3e-4db6-9043-f27b6d6e48a3
     文字数: 984
     内容: 会計情報レポート

有形固定資産に関連する会計基準等のまとめ

品質管理本部　会計監理部 公認会計士　宮(cid:8443) 徹 品質管理本部　会計監理部 公認会計士　廣瀬由美子

Toru Miyazaki

Yumiko Hirose

品質管理本部 会計監理部において、会計処 理および開示に関して相談を受ける業務、な らびに研修・セミナー講師を含む会計に関す る当法人内外への情報提供などの業...
     メタデータ: {'source': '/Users/iwamurahayato/myproject/astena-personal/document/docs_for_index/ey-japan-info-sensor-2023-06-03.pdf'}

  📑 ドキュメント 2:
     ID: d265a156-21af-42ce-9bb4-79407681e54d
     文字数: 989
     内容: どの会計基準等を参照するかという点が非常に分かり

づらくなっていることが原因ではないかと思います。

そこで、本稿では、有形固定資産に関連して会計処

7. 土地 8. リース資産（財務諸表提出会社がファイナン ス・リース取引におけるリース物件の借主で

理が生じる場面ごとにどの会計基準等を参考にすべき

かを